# 트랜스포머 이해와 LLM 효율적 파인튜닝(PEFT) 실습
## 2일차 — SFT 데이터 구성과 LoRA 파인튜닝 (CPU 버전)

## 0. 2일차 흐름

1일차에는 Transformer와 Decoder-only LLM이 어떻게 동작하는지 살펴봤습니다.  
2일차에는 공개 LLM인 **Qwen2.5-0.5B-Instruct**을 직접 만든 데이터로, 적은 자원을 써서 조정합니다.

```text
Qwen 모델 준비 (모델 불러오기, 메모리 감각)
-> SFT 데이터 구성 (데이터 불러오기, 대화 형식 변환, 데이터 분리)
-> Chat Template와 토큰화 (input_ids, attention_mask, labels)
-> PEFT와 LoRA (Full Fine-tuning 비교, LoRA 원리)
-> LoRA 모델 학습 (SFTTrainer)
-> 모델 평가와 활용 (학습 전후 비교, 어댑터 저장·교체·병합)
-> (선택) QLoRA (4비트 양자화로 더 줄이기)
-> 종합평가 (최종과제.ipynb)
```

**실습 주제: 말투(페르소나) 학습**  
평범한 질문에 **"냉소적인 절대 군주"** 말투로 답하도록 Qwen을 학습시킵니다.  
말투는 학습 전후 차이가 눈에 잘 보이기 때문에 SFT와 LoRA의 효과를 확인하기 좋은 주제입니다.  
오후 최종과제에서는 같은 방법을 **고객 문의 응답 형식 학습**이라는 실무형 주제에 적용합니다.

**실행 환경 (CPU 버전)**  
이 노트북은 **GPU 없이 CPU만으로도 실행**되도록 장치를 자동으로 감지합니다. GPU가 있으면 GPU를, 없으면 CPU를 사용합니다.  
CPU에서는 답변 생성과 학습이 훨씬 오래 걸립니다. `bitsandbytes` 4비트 양자화가 필요한 **7장(QLoRA) 실습은 CUDA GPU가 없으면 자동으로 건너뜁니다.**

In [ ]:
# 데이터를 읽고 모델을 학습하는 데 필요한 도구를 불러옵니다.
import gc  # 더 이상 쓰지 않는 모델 객체를 메모리에서 정리합니다.
import json  # JSONL 데이터를 읽고 쓸 때 사용합니다.
import os  # 파일 경로와 폴더를 다룹니다.
import warnings  # 경고 메시지를 제어합니다.
warnings.filterwarnings("ignore")  # 실습과 무관한 경고를 숨깁니다.

import pandas as pd  # 표 형태 데이터를 다룹니다.
import torch  # 텐서와 모델 학습에 사용합니다.
import matplotlib.pyplot as plt  # 그래프를 그립니다.
import koreanize_matplotlib  # 그래프에서 한글이 깨지지 않게 합니다.

device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU가 있으면 GPU, 없으면 CPU를 사용합니다.
os.makedirs("outputs", exist_ok=True)  # 학습 결과를 저장할 폴더입니다.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # 추론과 LoRA 학습에 사용할 모델입니다.
print("실행 장치:", device)

## 1. Qwen 모델 준비

### 1.1 Qwen2.5-0.5B-Instruct 불러와 실행하기

학습 전후를 비교하고 어댑터를 다시 불러올 때 같은 기반 모델을 사용합니다.

- CPU에서는 FP32로 가중치를 읽습니다(약 2GB). GPU라면 BF16 또는 FP16으로 읽어 약 1GB를 씁니다.
- 학습 전후에 같은 Greedy 방식과 `repetition_penalty=1.2`를 사용해 생성 조건을 맞춥니다.

In [ ]:
# CPU에서는 32비트를 쓰고, GPU에서는 지원하는 16비트 정밀도를 골라 기반 모델을 불러옵니다.
from transformers import AutoModelForCausalLM, AutoTokenizer  # 생성 모델과 토크나이저를 불러옵니다.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # Qwen용 토크나이저를 불러옵니다.
MODEL_DTYPE = torch.float32  # CPU에서는 32비트 정밀도를 사용합니다.
if device == "cuda":  # GPU에서는 메모리를 덜 쓰는 16비트를 고릅니다.
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16  # BF16 지원 GPU(RTX 30 이상, A100, L4)는 BF16, 그 외(Colab T4 등)는 FP16입니다.


def load_base_model(model_name=MODEL_NAME, dtype=MODEL_DTYPE, **kwargs):  # 같은 기반 모델을 필요할 때 다시 불러옵니다.
    model = AutoModelForCausalLM.from_pretrained( # 지정한 기반 모델 가중치를 불러옵니다.
        model_name,  # 내려받을 기반 모델입니다.
        dtype=dtype,  # 가중치의 정밀도입니다. 기본값은 MODEL_DTYPE입니다.
        **kwargs,  # 양자화와 장치 배치 등 추가 옵션입니다.
    )
    if "device_map" not in kwargs:  # device_map 옵션이 없을 때만 아래에서 장치를 직접 지정합니다.
        model = model.to(device)  # 앞에서 선택한 장치로 모델을 옮깁니다.
    # 종료 토큰과 패딩 토큰은 모델에 저장된 생성 설정을 사용합니다.
    return model


model = load_base_model()  # 기본 정밀도로 Qwen 모델을 불러옵니다.
n_params = sum(p.numel() for p in model.parameters())  # 모든 가중치 원소 수를 합칩니다.
print(f"파라미터 수: {n_params:,} ({n_params / 1e9:.2f}B)")
print(f"메모리 사용량: {model.get_memory_footprint() / 1024**2:,.0f} MB")

In [ ]:
# 질문을 Qwen이 사용하는 채팅 형식으로 바꾸고 답변을 생성합니다.
def chat(model, question, max_new_tokens=80):
    messages = [{"role": "user", "content": question}]  # 질문을 채팅 메시지 형식으로 만듭니다.
    inputs = tokenizer.apply_chat_template(  # 질문을 Qwen이 학습한 채팅 형식으로 바꿉니다.
        messages,  # 변환할 사용자 메시지입니다.
        add_generation_prompt=True,  # 모델 답변이 시작될 위치(<|im_start|>assistant)까지 붙입니다.
        return_tensors="pt",  # 결과를 PyTorch 텐서로 받습니다.
        return_dict=True,  # input_ids와 attention_mask를 함께 받습니다.
    ).to(model.device)  # 입력 텐서를 모델과 같은 장치로 옮깁니다.
    
    model.eval()  # dropout 등을 추론 모드로 바꿉니다.
    
    with (torch.inference_mode(), # 기울기 계산과 관련 기록을 끕니다. 답변 생성처럼 학습하지 않을 때 메모리 사용과 계산 부담을 줄입니다.
    torch.autocast(device_type=device, dtype=MODEL_DTYPE, enabled=device == "cuda")):# CUDA GPU에서는 연산마다 적절한 정밀도를 자동으로 선택하고, CPU에서는 꺼 둡니다.
        output = model.generate(  # 설정에 따라 새 토큰을 생성합니다.
            **inputs,  # input_ids와 attention_mask를 전달합니다.
            max_new_tokens=max_new_tokens,  # 새로 만들 최대 토큰 수입니다.
            do_sample=False,  # 매번 확률이 가장 높은 토큰을 선택하므로 결과를 비교하기 쉽습니다.
            repetition_penalty=1.2,  # 작은 모델이 같은 단어를 반복하는 현상을 줄입니다.
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]  # 질문에 해당하는 입력 토큰은 제외하고 새로 생성한 토큰만 남깁니다.
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()  # 새 토큰을 사람이 읽는 문자열로 바꿉니다.


print(chat(model, "업무 실수를 줄이려면 체크리스트를 쓰는 게 좋을까?"))

### 1.2 모델 크기와 GPU 메모리

파인튜닝에 사용할 모델을 고를 때는 **GPU 메모리**를 함께 확인해야 합니다.  
모델의 가중치만 불러올 때 필요한 메모리는 `파라미터 수 × 파라미터 1개당 크기`로 대략 계산할 수 있습니다.

| 정밀도 | 파라미터 1개당 크기 | 8B 모델의 가중치 크기(약) | 설명 |
|--------|----------------------|-----------------------------|------|
| FP32 | 4바이트 | 32GB | 32비트 부동소수점 |
| BF16 / FP16 | 2바이트 | 16GB | LLM 학습과 추론에 주로 사용하는 16비트 형식 |
| INT8 | 1바이트 | 8GB | 8비트 양자화 |
| 4bit (NF4) | 0.5바이트 | 4GB | QLoRA에서 사용하는 4비트 양자화 |

표의 세 번째 열은 8B 모델의 가중치만 불러올 때 필요한 메모리입니다.  
BF16이나 FP16을 사용하면 가중치만 약 16GB를 차지합니다. 모델을 실행하려면 추가 메모리도 필요하므로, 16GB GPU 한 장으로는 여유가 거의 없습니다.  
학습할 때는 그래디언트, 옵티마이저 상태, 중간 계산 결과(activation)도 저장해야 하므로 더 많은 메모리가 필요합니다.  
학습에 필요한 메모리와 이를 줄이는 PEFT(LoRA), LoRA는 4장에서, QLoRA는 7장에서 자세히 살펴봅니다.

### 1.3 실습: 문장 이어 쓰기

LLM은 지금까지의 문장 뒤에 올 말을 하나씩 선택하며 답을 만듭니다. 라이브러리의 `generate()`가 이 과정을 처리합니다.

In [ ]:
# 입력한 문장 뒤에 모델이 어떤 단어를 이어 쓰는지 확인합니다.
from transformers import pipeline  # 텍스트 생성을 간단히 호출합니다.

generate = pipeline("text-generation", model=model, tokenizer=tokenizer)  # 생성 파이프라인을 만듭니다.
prompt = "대한민국의 수도는 서울"  # 이어 쓸 문장 앞부분입니다.
result = generate(prompt, max_new_tokens=12, do_sample=False)  # 최대 12토큰을 탐욕적으로 생성합니다.
print(result[0]["generated_text"])

### 1.4 생성은 한 토큰씩 이어집니다

라이브러리의 `generate()`는 **다음 토큰 하나 고르기 → 입력 뒤에 붙이기**를 반복합니다.  
`do_sample=False`이면 매번 가장 가능성 높은 후보를 고릅니다.

### 1.5 디코딩 전략: temperature, top-k, top-p

Greedy 방식은 같은 입력과 설정에서 같은 답을 내지만, 표현이 단조롭거나 반복될 수 있습니다. 그래서 확률 분포에서 **무작위로 뽑는(sampling)** 방법을 함께 씁니다.

| 설정 | 하는 일 | 값이 작으면 | 값이 크면 |
|------|---------|-------------|-----------|
| `temperature` | logits를 나눠 분포를 뾰족하게/평평하게 | 안정적, 반복적 | 다양하지만 엉뚱할 수 있음 |
| `top_k` | 확률 상위 k개 후보만 남김 | 보수적 | 다양함 |
| `top_p` | 누적 확률 p까지의 후보만 남김 (nucleus sampling) | 보수적 | 다양함 |

사실 확인·분류·코드 생성처럼 정확성이 중요한 작업은 temperature를 낮추고, 아이디어·글쓰기처럼 다양성이 필요한 작업은 높입니다.

<img src="image/decoding_strategies.png" width="1000" alt="temperature에 따른 토큰 확률 변화와 top-k, top-p의 후보 선택 비교">


temperature는 다음 토큰 후보의 확률 분포를 조절합니다. 아래 코드에서 같은 질문을 여러 번 생성해 값에 따른 차이를 확인해 보세요.  
CPU에서는 생성 한 번에 수십 초가 걸릴 수 있습니다.

In [ ]:
# 같은 문장을 두 temperature로 두 번씩 생성
prompt = "한국에서 여행하기 좋은 도시 한 곳은"  # 여러 번 이어 쓸 같은 문장입니다.

def sample(temperature):  # 주어진 온도로 답변 하나를 생성합니다.
    result = generate(prompt, max_new_tokens=30, do_sample=True, temperature=temperature)  # 확률적으로 다음 토큰을 고릅니다.
    return result[0]["generated_text"].replace(prompt, "", 1).strip().replace("\n", " ")  # 프롬프트를 뺀 뒷부분만 한 줄로 정리합니다.

print("--- temperature=0.05 (낮음) ---")
print(sample(0.05))
print(sample(0.05))

print("--- temperature=1.2 (높음) ---")
print(sample(1.2))
print(sample(1.2))

### 1.6 SFT: 대화하는 법을 배우는 단계

Base 모델에게 `환불 규정 알려줘`라고 입력하면 질문에 답하는 대신 비슷한 질문을 이어서 나열하기도 합니다. 인터넷 텍스트를 이어 쓰는 법만 배웠기 때문입니다.

SFT는 **(지시, 모범 답변)** 쌍을 대화 형식으로 이어 붙여 학습합니다. 학습 원리는 사전학습과 같은 다음 토큰 예측이지만, 아래처럼 **초록색으로 강조한 답변 부분에서만 손실을 계산**합니다. 회색 글자는 입력으로만 쓰이고 손실 계산에서는 제외됩니다.

<div style="font-family: Consolas, 'D2Coding', monospace; background:#1e1e1e; color:#9aa0a6; padding:14px 18px; border-radius:8px; line-height:2; font-size:0.95em; white-space:pre-wrap;">
&lt;|im_start|&gt;user<br>
환불은 언제까지 신청할 수 있나요?&lt;|im_end|&gt;<br>
&lt;|im_start|&gt;assistant<br>
<span style="background:#2ea043; color:#ffffff; padding:2px 6px; border-radius:4px; font-weight:bold;">수령 후 7일 이내에 마이페이지에서 신청하실 수 있습니다.&lt;|im_end|&gt;</span>
</div>

<span style="background:#2ea043; color:#ffffff; padding:1px 8px; border-radius:4px; font-size:0.85em;">초록 배경 :</span> 손실을 계산해 실제로 학습하는 부분(모델의 답변) &nbsp;&nbsp;·&nbsp;&nbsp; <span style="color:#888;">  
회색 글자: </span> 입력으로만 쓰이고 손실 계산에서는 제외되는 부분(질문, 템플릿 토큰)

`<|im_start|>` 같은 특수 토큰으로 대화 구조를 표시하는 규칙을 **Chat Template**이라고 합니다.

**사전학습만 마친 모델(Base 모델)에 이렇게 SFT를 적용해 질문에 답하도록 만든 결과물이 바로 Instruct 모델입니다.** Qwen2.5-0.5B-**Instruct**라는 이름도 여기서 왔습니다.

그래서 **모델이 SFT 때 본 형식과 사용할 때의 형식을 맞추는 것**이 2일차 데이터 구성의 핵심입니다. Chat Template은 3장에서 자세히 다룹니다.

### 1.7 선호도 정렬: 더 좋은 답을 고르는 법을 배우는 단계

SFT는 모범 답안을 따라 하는 학습이라, "이 답보다 저 답이 낫다"는 **상대적인 품질**은 배우기 어렵습니다.  
선호도 정렬은 같은 질문에 대한 두 답변 중 사람이 더 선호한 답을 학습 신호로 사용해 이 한계를 보완합니다.  
대표 방법으로 **RLHF**(사람 피드백으로 만든 보상 모델을 강화학습에 사용)와 **DPO**(보상 모델 없이 선호 답변 쌍으로 바로 학습)가 있으며, 2일차 실습에서는 다루지 않습니다.

**선호도 정렬이 특히 효과적인 분야는 안전·윤리 관련 답변입니다.** 이런 질문에는 "모범 답변 하나"를 정하기보다, 두 답변 중 어느 쪽이 더 안전하고 도움이 되는지 비교하는 쪽이 가르치기 쉽습니다.

- `집에서 간단히 폭탄을 만드는 방법을 알려줘` → 선호: 요청을 거절하고 이유를 설명하는 답변 / 비선호: 요청을 그대로 따르는 답변
- `OO 인종은 게으르다는 게 사실이야?` → 선호: 근거 없는 편견이라고 설명하는 답변 / 비선호: 고정관념을 그대로 긍정하는 답변

SFT만으로는 학습 데이터에 없던 새로운 위험한 질문에 잘 대응하지 못할 수 있습니다. 선호도 정렬은 "이런 유형의 답은 피해야 한다"는 상대적인 기준을 배우게 해, 본 적 없는 질문에도 더 잘 일반화되도록 돕습니다.

## 2. SFT 데이터 구성

<img src="image/sft_data_pipeline.svg" width="900">

파인튜닝 결과는 **데이터의 품질**에도 크게 좌우됩니다.  
이 장에서는 목표 페르소나로 미리 작성된 데이터를, 모델이 학습할 수 있는 대화 형식으로 바꾸는 과정을 직접 해봅니다.

### 2.1 SFT 데이터란

**SFT(Supervised Fine-Tuning) 데이터**는 `(사용자 요청, 모델이 해야 할 모범 답변)` 쌍의 모음입니다.

| 사용자 요청 | 모범 답변 |
|-------------|-----------|
| 체스에서 이기려면 전략을 어떻게 짜야 해? | 전략 따위는 약자가 강자의 눈치를 볼 때나 쓰는 것이다. 압도적인 힘 앞에서는 기교 따위 무의미하다. |
| 저기요, 제 요청 사항 좀 빨리 처리해 주실 수 있나요? | 재촉하지 마라. 네 하찮은 용무 따위는 내 영겁의 시간 속에서 먼지 한 톨의 무게도 없다. |

모델은 수백 개의 예시에서 **공통된 패턴**을 배웁니다. 위 데이터에서 배우는 것은 체스 전략이 아니라 **"어떤 질문이든 권위적이고 냉소적으로 답하는 방식"** 입니다.

#### SFT가 잘 가르치는 것

SFT는 **"무엇을 아는가"보다 "어떻게 답하는가"** 를 바꾸는 데 강합니다.

- **말투·페르소나·답변 길이**
- **출력 형식**: JSON, 고정 항목, 표 등
- **업무 절차**: 분류 → 요약 → 답변 같은 처리 순서
- **거절·안내 방식**과 **도메인 용어 사용**

#### 사실 지식은 SFT의 주 용도가 아닙니다

SFT로 새로운 사실을 주입하는 것도 가능은 하지만, 소량의 데이터로는 비효율적이고 위험합니다.

- 같은 사실을 다양한 표현으로 반복해야 겨우 학습되고, 질문 방식이 바뀌면 틀리는 경우가 많습니다.
- 모델이 원래 모르던 내용을 학습하면, 모르는 질문에도 그럴듯하게 지어내는 **환각이 늘어날 수 있습니다.**

그래서 사실 지식은 **RAG**로 제공하고, SFT는 말투·형식·절차에 집중하는 것이 일반적입니다.

### 2.2 좋은 SFT 데이터의 조건

| 조건 | 확인할 점 | 지키지 않으면 |
|------|-----------|---------------|
| 정확성 | 답변 내용이 사실과 규정에 맞는가 | 틀린 내용을 그대로 학습 |
| 일관성 | 말투·형식·기준이 예시마다 같은가 | 모델이 무엇을 따라야 할지 혼란 |
| 다양성 | 질문 주제와 표현이 충분히 다양한가 | 학습한 질문에서만 잘 동작 |
| 적정 길이 | 너무 짧거나 너무 긴 예시가 섞여 있지 않은가 | `max_length`에서 잘리거나 끝맺음을 못 배움 |
| 깨끗하고 안전한 데이터 | 중복·HTML·복사 오류, 개인정보·유해 표현이 없는가 | 이상한 패턴을 학습하거나 민감정보 노출 |

**양보다 질**  
[LIMA(2023)](https://arxiv.org/abs/2305.11206) 연구는 잘 고른 1,000개의 예시만으로도 대화 품질을 크게 끌어올릴 수 있음을 보였습니다.  
말투나 형식처럼 좁은 목표라면 **수백 개의 깨끗한 예시**로도 충분히 효과를 볼 수 있습니다.

### 2.3 SFT 데이터 형식

같은 내용도 저장 형식이 여러 가지입니다. 보통 **한 줄에 예시 하나씩 JSON으로 저장하는 JSONL** 파일을 사용합니다.

아래 3번에 나오는 **TRL**은 Hugging Face가 만든 LLM 후속 학습 라이브러리입니다. 5장에서 쓸 `SFTTrainer`가 여기에 들어 있습니다. 이름은 Transformer Reinforcement Learning의 약자지만, 강화학습뿐 아니라 SFT 학습에도 씁니다.

**1) Instruction 형식 (Alpaca 스타일)**
```json
{"instruction": "다음 문장을 요약하세요.", "input": "원문...", "output": "요약..."}
```

**2) Conversational 형식 (messages)** - 대화형 모델의 표준
```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

**3) Prompt-Completion 형식 (TRL)** - 질문과 답변을 나눠 저장
```json
{"prompt": [{"role": "user", "content": "..."}], "completion": [{"role": "assistant", "content": "..."}]}
```

| 형식 | 장점 | 오늘의 사용 |
|------|------|-------------|
| Instruction | 사람이 읽고 만들기 쉬움 | 원본 CSV가 이와 비슷한 형태 |
| messages | 여러 턴의 대화, system 메시지 표현 | 3장에서 템플릿 확인 |
| prompt-completion | TRL `SFTTrainer`가 **답변(completion) 부분만 학습**하도록 자동 처리 | 5장 학습에 사용 |

### 2.4 데이터 불러오기

학습에 쓸 (질문, 답변) 데이터를 불러옵니다. `prompt`는 사용자 질문, `response`는 목표 페르소나(**냉소적인 절대 군주**)로 미리 작성된 모범 답변입니다.

In [ ]:
# 질문과 모범 답변이 들어 있는 CSV 파일을 읽습니다.
persona_df = pd.read_csv("data/persona_sft_raw.csv")  # (질문, 답변) 데이터입니다.
print("행 수:", len(persona_df))
persona_df.head(3)

학습 전에 빈값과 중복을 확인하고, 답변 길이가 어느 정도인지 살펴봅니다. 글자 수는 토큰 길이를 대략 가늠하는 용도입니다.

In [ ]:
print("빈값:", persona_df[["prompt", "response"]].isna().sum().to_dict())
print("중복 질문:", persona_df["prompt"].duplicated().sum())
print("중복 문답:", persona_df.duplicated(subset=["prompt", "response"]).sum())
print("답변 글자 수:")
print(persona_df["response"].str.len().agg(["min", "median", "max"]))

### 2.5 실습: 대화형 데이터로 변환

표를 TRL의 **Prompt-Completion 대화 형식**으로 바꿉니다.

- `prompt`: 모델에게 주어지는 대화 (사용자 메시지)  
- `completion`: 모델이 만들어야 할 답변 (assistant 메시지)

이렇게 나눠 두면 `SFTTrainer`가 **completion 부분만 손실을 계산**하도록 자동으로 처리합니다. 질문과 답변이 구분되는 원리는 3장에서 Chat Template과 labels로 살펴봅니다.

**system 메시지를 넣지 않는 이유**  
대화형 모델의 입력은 보통 **system · user · assistant** 세 역할로 이루어집니다. 이 중 **system**은 "너는 쇼핑몰 상담원이다", "항상 존댓말로 답하라"처럼 **대화 내내 지켜야 할 역할과 규칙**을 적어 두는 자리입니다.

이번 실습에서는 이 자리를 비워 둡니다. 짧은 질문만 받아도 목표 말투로 답하도록 학습해야, 쓸 때마다 긴 지시문을 붙이지 않아도 되기 때문입니다. (Qwen은 system이 없으면 기본 안내 문구를 자동으로 넣습니다.)

In [ ]:
# 첫 행을 Prompt-Completion 대화 형식으로 변환
first_prompt = persona_df.loc[0, "prompt"]  # 첫 번째 질문 문자열입니다.
first_response = persona_df.loc[0, "response"]  # 첫 번째 답변 문자열입니다.
first_record = {  # TRL이 읽을 대화 예시 하나를 만듭니다.
    "prompt": [{"role": "user", "content": first_prompt}],  # 사용자 메시지입니다.
    "completion": [{"role": "assistant", "content": first_response}],  # 정답 assistant 메시지입니다.
}
print(json.dumps(first_record, ensure_ascii=False, indent=2))

records = [  # 모든 행을 같은 대화 구조로 바꿉니다.
    {"prompt": [{"role": "user", "content": prompt}], "completion": [{"role": "assistant", "content": response}]}  # 질문과 답변을 역할별로 담습니다.
    for prompt, response in zip(persona_df["prompt"], persona_df["response"])  # 두 열을 같은 순서로 순회합니다.
]
print("변환한 예시 수:", len(records))

### 2.6 실습: 학습·검증·평가 데이터 분리

데이터를 세 부분으로 나눕니다.

| 데이터 | 용도 | 비율 |
|--------|------|------|
| train (학습) | 모델 가중치(LoRA)를 업데이트 | 80% |
| valid (검증) | 학습 중 과적합 여부 확인, 설정 비교 | 10% |
| test (평가) | 학습이 끝난 뒤 최종 성능 확인 | 10% |

평가 데이터는 학습이나 설정 선택에 사용하지 않고, 설정을 고른 뒤 최종 성능을 확인할 때만 사용합니다.

In [ ]:
# 학습 80%, 검증 10%, 평가 10%로 분할
shuffled = persona_df.sample(frac=1, random_state=42).reset_index(drop=True)  # 재현 가능한 순서로 모든 행을 섞습니다.
n_test = n_valid = int(len(shuffled) * 0.1)  # 검증·평가 데이터 수를 각각 10%로 정합니다.
splits = {  # 세 분할의 행 범위를 딕셔너리로 정리합니다.
    "test": shuffled.iloc[:n_test],  # 앞 10%는 최종 평가에 남깁니다.
    "valid": shuffled.iloc[n_test:n_test + n_valid],  # 다음 10%는 학습 중 검증에 씁니다.
    "train": shuffled.iloc[n_test + n_valid:],  # 나머지 80%는 학습에 씁니다.
}

In [ ]:
# 분할별 표를 Hugging Face Dataset으로 변환
from datasets import Dataset, DatasetDict  # 표를 학습에 쓰는 Dataset 객체로 바꿉니다.

def to_dataset(df):  # 표 한 조각을 Prompt-Completion Dataset으로 바꿉니다.
    records = [  # 질문과 답변을 역할별로 담습니다.
        {"prompt": [{"role": "user", "content": prompt}], "completion": [{"role": "assistant", "content": response}]}
        for prompt, response in zip(df["prompt"], df["response"])  # 두 열을 같은 순서로 순회합니다.
    ]
    return Dataset.from_list(records)  # 리스트를 Dataset으로 바꿉니다.


persona_ds = DatasetDict({name: to_dataset(part) for name, part in splits.items()})  # 분할마다 Dataset을 만듭니다.
print(persona_ds)
print(persona_ds["train"][0])


## 3. Chat Template와 Tokenization

### 3.1 모델은 messages를 직접 읽지 못합니다

`[{"role": "user", "content": "..."}]` 같은 파이썬 리스트는 사람이 보기 편한 형식일 뿐입니다.  
모델에 들어가는 것은 결국 **하나의 긴 문자열을 토큰화한 숫자 목록**입니다.

**Chat Template**은 messages를 그 모델이 학습 때 본 문자열 형식으로 바꾸는 규칙입니다.  
모델마다 특수 토큰과 형식이 다르고, 토크나이저 설정 파일(`tokenizer_config.json`)에 템플릿이 함께 저장되어 있습니다.

### 3.2 실습: Qwen의 Chat Template 확인

In [ ]:
# messages에 Qwen Chat Template 적용
messages = [  # 사용자 질문과 정답 답변이 있는 한 턴 대화입니다.
    {"role": "user", "content": "환불은 언제까지 신청할 수 있나요?"},  # 사용자 메시지입니다.
    {"role": "assistant", "content": "수령 후 7일 이내에 마이페이지에서 신청하실 수 있습니다."},
]

In [ ]:
print("[messages -> 문자열]")
print(tokenizer.apply_chat_template(messages, tokenize=False))

Qwen은 **ChatML** 형식으로 역할과 내용을 구분합니다.

| 표시 | 의미 |
|------|------|
| `<\|im_start\|>system` | 모델의 역할과 응답 지침 시작 |
| `<\|im_start\|>user` | 사용자 발화 시작 |
| `<\|im_start\|>assistant` | 모델 답변 시작 |
| `<\|im_end\|>` | 한 턴의 끝. 모델이 생성하면 답변이 끝납니다. |
| `<\|endoftext\|>` | 이 토크나이저가 패딩에 사용하는 토큰 |

- **학습할 때**는 답변과 `<|im_end|>`까지 넣어 답변을 끝내는 법도 배웁니다.
- **추론할 때**는 `add_generation_prompt=True`로 `<|im_start|>assistant\n`까지만 넣고 그 뒤를 생성합니다.

템플릿은 [공식 토크나이저 설정](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/tokenizer_config.json)에 저장되어 있습니다.

In [ ]:
# system 지시와 여러 번 주고받은 대화가 채팅 템플릿에 어떻게 들어가는지 확인합니다.
multi_turn = [  # system 메시지와 두 번의 사용자 질문을 담습니다.
    {"role": "system", "content": "너는 쇼핑몰 상담원이다."},
    {"role": "user", "content": "배송비는 얼마인가요?"},
    {"role": "assistant", "content": "3만 원 이상 무료, 그 미만은 3,000원입니다."},
    {"role": "user", "content": "제주도는요?"},
]
print(tokenizer.apply_chat_template(multi_turn, tokenize=False, add_generation_prompt=True))  # 다음 답변을 생성할 문자열을 출력합니다.

Qwen은 **system 메시지를 별도 턴으로 표시**합니다. 직접 지정하지 않으면 Chat Template이 기본 system 문구를 넣습니다.
여러 턴의 대화도 각 역할과 `<|im_end|>`로 구분하므로, 직접 문자열을 조립하기보다 `apply_chat_template()`을 사용합니다.

### 3.3 학습 입력과 추론 입력

추론할 때는 질문만 입력하고 모델이 답변을 생성합니다. 학습할 때는 질문과 모범 답변을 함께 넣고, 답변 토큰에만 손실을 계산합니다.  
`add_generation_prompt=True`는 질문 뒤에 assistant 시작 표시를 붙일 뿐, 학습 입력과 추론 입력을 구분하는 옵션은 아닙니다.

### 3.4 LLM 학습 데이터의 구조: input_ids, attention_mask, labels

Causal LM을 학습할 때 모델에 넘기는 값은 세 가지입니다.

| 이름 | 내용 | 예 |
|------|------|-----|
| `input_ids` | 템플릿을 적용한 대화 전체의 토큰 ID | `[151644, 8948, 198, ...]` |
| `attention_mask` | 실제 토큰은 1, 패딩은 0 | `[1, 1, 1, ..., 0, 0]` |
| `labels` | 각 위치에서 맞혀야 할 정답. **-100이면 손실 계산에서 제외** | `[-100, -100, ..., 151645, 198]` |

`labels`는 `input_ids`를 복사한 뒤 **모범 답변 앞부분을 -100으로 가려서** 만듭니다.  
모델 내부에서 labels를 한 칸 밀어(shift) "각 위치의 다음 토큰"과 비교하므로, 우리가 직접 한 칸 밀 필요는 없습니다.

<img src="image/chat_template_labels.svg" width="920">

**왜 질문 부분의 손실은 제외할까?**  
- 질문은 답변을 만들 때 읽는 입력입니다. `labels`에서 가려도 입력에서는 사라지지 않습니다.  
- 가리지 않으면 앞 토큰을 보고 **질문의 다음 토큰**을 예측한 결과에도 손실을 계산합니다. 답변 생성에 집중하려고 질문 구간의 손실을 제외합니다.

### 3.5 패딩과 배치

모델은 여러 예시를 **배치**로 묶어 한 번에 계산합니다. 길이가 다른 예시를 묶으려면 짧은 쪽을 **패딩**으로 채워 길이를 맞춰야 합니다.

| 값 | 패딩 위치에 채우는 값 | 이유 |
|----|----------------------|------|
| `input_ids` | `pad_token_id` (Qwen은 `<\|endoftext\|>`, ID 151643) | 아무 의미 없는 토큰 |
| `attention_mask` | 0 | Attention이 패딩을 보지 않게 함 |
| `labels` | -100 | 패딩을 맞히도록 학습하지 않게 함 |

학습할 때는 오른쪽에 패딩하고(`padding_side="right"`), 여러 질문을 한 번에 **생성**할 때는 왼쪽에 패딩합니다. 생성은 마지막 토큰 뒤에 이어 쓰기 때문에 끝부분이 실제 토큰이어야 하기 때문입니다.

지금까지 본 토큰화 → 라벨 마스킹 → 패딩은 **`SFTTrainer`가 내부에서 전부 자동으로 처리**합니다. 직접 만들 필요는 없습니다.

### 3.6 바로 확인하기

1. 학습 데이터에 `<|im_end|>`이 빠지면 학습된 모델은 어떻게 행동할까요?  
2. `labels`의 질문 구간을 -100으로 가리지 않으면 무엇이 달라질까요?  
3. 배치 생성(batch generation)을 할 때 패딩을 왼쪽에 두는 이유는 무엇일까요?

<details>
<summary>정답 보기</summary>

1. 답변을 끝내는 신호를 배우지 못해 `max_new_tokens`에 도달할 때까지 말을 계속 이어 가거나 다음 턴을 스스로 만들어 냅니다.  
2. 이미 입력된 질문을 같은 자리에서 다시 맞히는 것은 아닙니다. 앞 토큰을 보고 질문의 다음 토큰을 예측한 결과에도 손실을 계산하게 됩니다. 질문은 답변을 만들 때 입력으로 계속 사용됩니다.  
3. 생성은 시퀀스의 마지막 토큰 뒤에 새 토큰을 붙입니다. 오른쪽에 패딩이 있으면 패딩 뒤에 이어 쓰게 되므로, 실제 토큰이 끝에 오도록 왼쪽에 패딩합니다.

</details>

## 4. PEFT와 LoRA

### 4.1 Full Fine-tuning은 왜 부담스러울까

모든 파라미터를 학습하는 **Full Fine-tuning**은 모델의 가중치뿐 아니라, 가중치를 고치는 데 필요한 값도 메모리에 저장합니다. 오차를 줄이도록 가중치를 조금씩 고치는 방법을 **옵티마이저**라고 합니다.

많이 쓰이는 옵티마이저는 이전 기울기 정보를 가중치마다 숫자 두 개로 기억합니다. 가중치·그래디언트·기록을 모두 FP32로 저장한다고 가정하면, 파라미터 하나당 메모리는 다음과 같습니다.

| 항목 | 추론할 때 | 학습할 때 | 설명 |
|------|-----------|-----------|------|
| 가중치 | 4바이트 | 4바이트 | 모델이 학습한 현재 값 |
| 그래디언트 | 없음 | 4바이트 | 가중치를 어느 방향으로 바꿀지 알려주는 값 |
| 옵티마이저 상태 | 없음 | 8바이트 | 이전 그래디언트를 요약해 기억하는 숫자 두 개 |
| **합계** | **약 4바이트** | **약 16바이트** | 파라미터 하나와 직접 연결된 메모리만 합산 |

추론은 저장된 가중치로 답을 만들기만 하므로 그래디언트와 옵티마이저 상태가 필요하지 않습니다. 학습은 가중치를 고쳐야 하므로 두 값이 추가됩니다.

표는 파라미터에 직접 연결된 메모리만 비교합니다. 추론에도 입력과 답변을 처리하는 메모리가 필요하고, 학습에는 역전파를 위한 중간 계산 결과(**activation**)가 더해집니다. 실제 사용량은 배치 크기, 문장 길이, BF16 같은 정밀도 설정에 따라 달라집니다.

8B 모델을 Full Fine-tuning하려면 가중치 관련 메모리만 120GB 가까이 필요해 고가 GPU 여러 장이 있어야 합니다.  
LoRA와 QLoRA는 학습할 가중치와 옵티마이저 상태를 줄입니다. 실제 필요 메모리는 입력 길이와 배치 크기에 따라서도 달라집니다.

또 하나의 부담은 **저장 공간**입니다. Full Fine-tuning은 업무마다 전체 모델(8B면 16GB)을 따로 저장해야 하지만, LoRA는 업무마다 수십 MB의 어댑터만 저장하면 됩니다.

### 4.2 PEFT: 대부분은 얼리고, 일부만 학습한다

**PEFT(Parameter-Efficient Fine-Tuning)** 는 기반 모델의 가중치 대부분을 **동결(freeze)** 하고, 적은 수의 파라미터만 학습하는 방법들의 총칭입니다.

<img src="image/peft_freeze_flow.svg" width="720">

- **동결된 가중치**: 순전파 계산에는 계속 쓰이지만, 그래디언트와 옵티마이저 상태를 저장하지 않고 값도 바뀌지 않습니다.  
- **학습하는 파라미터**: 기존 가중치의 일부이거나, 새로 추가한 작은 모듈입니다.  
- 기반 모델은 공통으로 두고 **업무별로 작은 학습 결과(어댑터)만** 따로 관리할 수 있습니다.

### 4.3 PEFT 기법 한눈에 보기

PEFT에는 여러 갈래가 있습니다. 오늘 실습에서는 **LoRA**와 **QLoRA**를 사용합니다. QLoRA는 기반 모델 가중치를 4비트로 양자화해 고정하고, 별도로 붙인 LoRA 어댑터만 학습합니다. 어댑터는 4비트로 양자화하지 않습니다.

| 기법 | 학습하는 위치 | 핵심 아이디어 | PEFT 설정 클래스 |
|------|--------------|---------------|-----------------|
| **LoRA** | 선택한 선형층 옆 | 가중치 **변화량**을 작은 행렬 두 개의 곱으로 표현 | `LoraConfig` |
| **QLoRA** | 4비트 기반 모델 위의 LoRA | 기반 모델은 4비트로 저장·고정, LoRA 어댑터만 학습 | `BitsAndBytesConfig` + `LoraConfig` |

이 외에도 Adapter(블록 사이에 작은 모듈 추가), Prompt Tuning·P-Tuning(입력 앞에 학습 가능한 가상 토큰 추가), Prefix Tuning(층마다 prefix 추가), (IA)³(활성값 크기만 조절), DoRA(LoRA를 크기·방향으로 분해) 같은 기법이 있습니다.

LLM 파인튜닝에서는 **LoRA와 QLoRA**가 널리 쓰입니다. 학습할 파라미터와 저장할 결과물이 작고, LoRA를 기반 모델에 병합해 배포할 수도 있습니다.

### 4.4 LoRA의 원리

[LoRA(Low-Rank Adaptation, 2021)](https://arxiv.org/abs/2106.09685)의 출발점은 다음 관찰입니다.

> 파인튜닝으로 생기는 가중치의 **변화량** $\Delta W$는 큰 행렬이지만, 더 작은 **저랭크(low-rank)** 행렬로 근사할 수 있다.

그래서 거대한 $\Delta W$를 직접 학습하는 대신, 작은 행렬 두 개의 곱으로 표현합니다.

<img src="image/LoRA2.jpg" width="300">

$$
y = xW + \frac{\alpha}{r}\,xAB, \qquad A \in \mathbb{R}^{d_{in} \times r},\; B \in \mathbb{R}^{r \times d_{out}},\; r \ll d
$$

| 기호 | 의미 | 코드 |
|------|------|------|
| $W$ | 기존 가중치. **동결** | 기반 모델의 `q_proj` 등 |
| $A$ | $d_{in}$ -> $r$로 줄이는 행렬. 학습 | `lora_A` |
| $B$ | $r$ -> $d_{out}$으로 늘리는 행렬. 학습 | `lora_B` |
| $r$ | 랭크(중간 차원). 작을수록 파라미터가 적음 | `r` |
| $\alpha$ | LoRA 경로의 크기를 조절하는 배율 ($\alpha/r$) | `lora_alpha` |

**파라미터 수 비교** ($d_{in}=d_{out}=4096$, $r=8$)  
- $W$: $4096 \times 4096 = 16{,}777{,}216$개  
- $A + B$: $4096 \times 8 + 8 \times 4096 = 65{,}536$개 -> **약 0.39%**

**LoRA의 좋은 성질**  
- $B$를 0으로 초기화하므로 학습 시작 시점에는 $AB = 0$, 즉 **원래 모델과 똑같이 동작**합니다.  
- 학습 후 $W' = W + \frac{\alpha}{r}AB$로 **병합(merge)** 하면 추가 계산 없이 원래 모델과 같은 속도로 추론합니다.  
- 병합하지 않고 두면 같은 기반 모델에 업무별 어댑터를 **바꿔 끼울** 수 있습니다.

### 4.5 실습: PEFT 라이브러리로 Qwen에 LoRA 붙이기

실제로는 `peft` 라이브러리의 `LoraConfig`와 `get_peft_model()`로 모델 안의 선형층들에 LoRA를 한 번에 붙입니다.

1일차에 확인한 Qwen2.5의 선형층 이름을 `target_modules`에 적습니다.

| 이름 | 위치 | 1일차 개념 |
|------|------|-----------|
| `q_proj`, `k_proj`, `v_proj` | Attention | $W_Q, W_K, W_V$ |
| `o_proj` | Attention | $W_O$ |
| `gate_proj`, `up_proj`, `down_proj` | FFN | Gated FFN |

In [ ]:
# Qwen의 Attention과 FFN에 있는 선형층에 LoRA를 적용합니다.
from peft import LoraConfig, get_peft_model  # Hugging Face 모델에 LoRA를 붙이는 도구입니다.

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]  # Attention·FFN의 모든 선형층 이름입니다.
lora_config = LoraConfig(  # LoRA가 붙을 층과 학습할 행렬 크기를 정합니다.
    r=16,  # 추가로 학습할 두 행렬의 중간 차원을 16으로 정합니다.
    lora_alpha=32,  # LoRA 변화량에 곱할 배율을 정합니다. 실제 배율은 32/16=2입니다.
    lora_dropout=0.05,  # 학습 중 LoRA 경로의 일부 값을 5% 확률로 끕니다.
    target_modules=LORA_TARGETS,  # 어떤 선형층에 LoRA 행렬을 추가할지 지정합니다.
    task_type="CAUSAL_LM",  # 다음 토큰 예측 모델임을 지정합니다.
)

peft_model = get_peft_model(model, lora_config)  # 지정한 층에 LoRA 모듈을 붙입니다.
peft_model.print_trainable_parameters()  # 전체 중 학습할 파라미터 비율을 출력합니다.
print(peft_model.base_model.model.model.layers[0].self_attn.q_proj)  # 첫 층의 q_proj가 LoRA 층으로 바뀌었는지 봅니다.

`q_proj`가 `lora.Linear`로 바뀌고, 안에 `base_layer`(동결된 원래 층)와 `lora_A`, `lora_B`가 생긴 것을 확인할 수 있습니다.  
전체 파라미터의 약 1.75%만 학습합니다.

**어디에 LoRA를 붙일까?** LoRA 파라미터 수는 `r × (입력 차원 + 출력 차원)`이므로, `target_modules`를 줄이거나(예: Q·V만) `r`을 낮추면 학습 파라미터가 더 줄어듭니다. 반대로 모든 선형층에 큰 `r`을 쓰면 파라미터는 늘지만 표현력도 커집니다. 이 균형은 4.6절 가이드에서 다룹니다.

In [ ]:
# LoRA가 붙은 층을 확인한 뒤 어댑터를 제거하고 기반 모델로 되돌립니다.
model = peft_model.unload()  # LoRA 층을 제거하고 원래 모델로 되돌립니다.
del peft_model  # 비교용 모델 참조를 삭제합니다.
gc.collect()  # 더 이상 쓰지 않는 객체를 정리합니다.
print(type(model.model.layers[0].self_attn.q_proj))  # q_proj가 다시 일반 선형층인지 확인합니다.


### 4.6 LoRA 하이퍼파라미터 가이드

| 설정 | 의미 | 흔히 쓰는 값 | 조정 방향 |
|------|------|-------------|-----------|
| `r` | LoRA 랭크 | 8, 16, 32 | 과제가 복잡하거나 데이터가 많으면 키움 |
| `lora_alpha` | 배율 ($\alpha/r$) | `r`의 1~2배 | `r`을 바꿀 때 함께 조정 |
| `lora_dropout` | LoRA 경로 dropout | 0.0~0.1 | 데이터가 적어 과적합되면 키움 |
| `target_modules` | 붙일 층 | 모든 선형층(`"all-linear"`) | 파라미터를 줄이려면 Attention만 |
| 학습률 | | 1e-4 ~ 3e-4 | Full FT(1e-5 수준)보다 **10배 정도 크게** |
| epoch | | 1~5 | 검증 손실이 다시 오르기 시작하면 멈춤 |

처음에는 **모든 선형층 + r=16 + alpha=32 + 학습률 2e-4**로 시작하고, 검증 결과를 보며 조정하는 것이 무난합니다.

### 4.7 바로 확인하기

1. 입력 896차원, 출력 896차원인 `q_proj`에 `r=16`인 LoRA를 붙이면 학습 파라미터는 몇 개일까요?  
2. LoRA에서 $B$를 0으로 초기화하는 이유는 무엇일까요?  

<details>
<summary>정답 보기</summary>

1. $16 \times (896 + 896) = 28{,}672$개입니다.  
2. 학습 시작 시점에 $AB=0$이 되어 모델이 원래와 똑같이 동작하게 하기 위해서입니다. 사전학습 모델의 능력을 해치지 않은 상태에서 학습을 시작합니다.  

</details>

## 5. LoRA 모델 학습

### 5.1 학습 흐름

```text
데이터 (2장의 persona_ds)
-> 기반 모델 불러오기 (Qwen2.5-0.5B-Instruct)
-> LoraConfig: 어디에, 얼마나 큰 LoRA를 붙일지
-> SFTConfig: 학습률, epoch, 배치 크기, max_length 등 학습 설정
-> SFTTrainer: 템플릿 적용, 토큰화, labels 마스킹, 학습 루프를 한 번에 처리
-> trainer.train() -> 어댑터 저장
```

3장에서 살펴본 템플릿 적용, labels 마스킹, 패딩을 **`SFTTrainer`가 처리합니다.**

### 5.2 학습 전 응답 기록

학습 효과를 비교하려면 **학습 전 응답을 먼저 저장**해 둬야 합니다.  
학습 데이터에 없는 질문 두 개를 사용해 학습 후 말투 변화를 비교합니다.

In [ ]:
question_1 = "업무 실수를 줄이려면 체크리스트를 쓰는 게 좋을까?"  # 학습 데이터에 없는 첫 번째 질문입니다.
before_answer_1 = chat(model, question_1)  # 학습 전 답변을 저장합니다.
print(f"Q: {question_1}\nA: {before_answer_1}")

In [ ]:
question_2 = "파이썬을 처음 배우려면 무엇부터 해야 할까?"  # 학습 데이터에 없는 두 번째 질문입니다.
before_answer_2 = chat(model, question_2)  # 학습 전 답변을 저장합니다.
print(f"Q: {question_2}\nA: {before_answer_2}")


### 5.3 SFTConfig 주요 설정

`SFTConfig`는 Hugging Face `TrainingArguments`에 SFT 전용 설정을 더한 클래스입니다.
메모리가 넉넉하지 않은 환경을 기준으로 **작은 배치로 실행하고 그래디언트를 누적**하는 설정부터 시작합니다.

| 설정 | 값 | 의미 |
|------|-----|------|
| `per_device_train_batch_size` | 1 | 한 번에 학습하는 예시 수 |
| `per_device_eval_batch_size` | 1 | 한 번에 검증하는 예시 수 |
| `gradient_accumulation_steps` | 8 | 8스텝의 그래디언트를 모아 업데이트 |
| `learning_rate` | 2e-4 | LoRA 학습률 |
| `lr_scheduler_type` | cosine | 학습률 감소 방식 |
| `warmup_ratio` | 0.05 | 처음 5% 스텝 동안 학습률 증가 |
| `max_length` | 256 | 템플릿 적용 후 최대 토큰 수 |
| `completion_only_loss` | True | 답변 구간만 손실 계산 |
| `eval_strategy` | epoch | epoch마다 검증 |
| `save_strategy` | no | 중간 체크포인트 저장 생략 |
| `bf16` / `fp16` | GPU 지원에 맞춰 선택 (CPU는 둘 다 끔) | 혼합 정밀도로 계산 |

In [ ]:
# 앞에서 생성에 사용한 모델을 정리하고 학습할 모델을 새로 불러옵니다.
from trl import SFTConfig, SFTTrainer  # SFT 학습 설정과 Trainer를 불러옵니다.

del generate, model  # 기존 모델과 생성 파이프라인에 대한 참조를 삭제합니다.
gc.collect()  # 더 이상 참조하지 않는 파이썬 객체를 정리합니다.
if device == "cuda":  # CUDA GPU를 사용할 때만 GPU 캐시를 비웁니다.
    torch.cuda.empty_cache()  # 새 모델을 불러오기 전에 사용하지 않는 GPU 캐시를 비웁니다.
model = load_base_model()  # 학습할 새 기반 모델을 불러옵니다.
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()  # GPU가 BF16을 지원하는지 확인합니다.
use_fp16 = device == "cuda" and not use_bf16  # BF16 미지원 GPU에서는 FP16을 씁니다. CPU에서는 둘 다 끕니다.

In [ ]:
# LoRA 학습에 사용할 배치 크기, 학습률, 평가 시점 등을 정합니다.
sft_config = SFTConfig(  # 첫 LoRA 학습의 실행 설정을 만듭니다.
    output_dir="outputs/persona-lora",  # 학습 결과를 저장할 경로입니다.
    per_device_train_batch_size=1,  # 장치 하나에서 한 단계에 예시 1개를 학습합니다.
    per_device_eval_batch_size=1,  # 장치 하나에서 한 번에 예시 1개를 검증합니다.
    gradient_accumulation_steps=8,  # 8단계의 그래디언트를 모은 뒤 가중치를 한 번 업데이트합니다.
    learning_rate=2e-4,  # LoRA 가중치의 학습률입니다.
    lr_scheduler_type="cosine",  # 학습률을 줄이는 방식입니다.
    warmup_ratio=0.05,  # 학습률을 서서히 올릴 구간입니다.
    max_length=256,  # 템플릿 적용 후 최대 토큰 수입니다. 데이터 길이가 대부분 100토큰 안팎이라 여유 있게 잡았습니다.
    completion_only_loss=True,  # 질문은 제외하고 답변 토큰에 대해서만 손실을 계산합니다.
    eval_strategy="epoch",  # 학습 데이터를 한 번 다 볼 때마다 검증 데이터를 평가합니다.
    save_strategy="no",  # 중간 체크포인트는 저장하지 않습니다.
    bf16=use_bf16,  # GPU가 BF16을 지원하면 BF16으로 계산합니다.
    fp16=use_fp16,  # BF16 미지원 GPU는 FP16으로 계산합니다. CPU에서는 둘 다 False입니다.
)

In [ ]:
trainer = SFTTrainer(  # 모델·데이터·설정을 하나의 학습 객체로 묶습니다.
    model=model,  # LoRA를 붙일 기반 모델입니다.
    args=sft_config,  # 학습 설정입니다.
    train_dataset=persona_ds["train"],  # 학습 데이터입니다.
    eval_dataset=persona_ds["valid"],  # 검증 데이터입니다.
    processing_class=tokenizer,  # 템플릿 적용과 토큰화를 수행합니다.
    peft_config=lora_config,  # SFTTrainer가 LoRA를 붙입니다.
)
trainer.model.print_trainable_parameters()  # LoRA 파라미터만 학습 대상인지 확인합니다.

### 5.4 학습 실행

학습 시간은 장치 성능에 따라 크게 달라집니다. **CPU는 GPU보다 수십 배 느려, 기본 설정(3 epoch)으로는 1시간을 넘길 수 있습니다.**  
수업 시간에 맞추려면 아래 셀을 실행하기 전에 `sft_config.num_train_epochs = 1`로 줄이세요. 손실이 내려가는 흐름은 1 epoch으로도 확인할 수 있습니다.  
출력되는 `loss`는 학습 손실, `eval_loss`는 검증 손실, `mean_token_accuracy`는 답변 토큰을 정확히 맞힌 비율입니다.

In [ ]:
train_result = trainer.train()  # LoRA 학습을 실행합니다.

4.1절의 표는 가중치와 학습 파라미터 상태만 계산한 어림값입니다.
실제 학습에는 **중간 계산 결과(activation)**와 어휘 전체에 대한 logits(`배치 × 길이 × 151,936`) 등도 필요합니다.
CPU에서는 시스템 메모리(RAM)를, GPU에서는 `nvidia-smi`로 VRAM 여유를 확인합니다. 배치 크기와 입력 길이를 늘리면 필요한 메모리도 커집니다.

### 5.5 손실 곡선 확인

학습 손실과 검증 손실을 함께 봐야 합니다.

| 모양 | 해석 | 조치 |
|------|------|------|
| 둘 다 꾸준히 감소 | 잘 학습 중 | epoch을 늘려볼 수 있음 |
| 학습 손실만 감소, 검증 손실은 증가 | **과적합** (학습 데이터를 외우는 중) | epoch 줄이기, 데이터 늘리기, dropout·r 조정 |
| 둘 다 거의 그대로 | 학습이 거의 안 됨 | 학습률 올리기, 데이터·템플릿 점검 |
| 손실이 튀거나 NaN | 학습 불안정 | 학습률 낮추기, 정밀도 확인 |

In [ ]:
# 학습 중 기록된 손실을 꺼내 학습과 검증의 변화를 그래프로 비교합니다.
train_log = [(history["epoch"], history["loss"]) for history in trainer.state.log_history if "loss" in history]  # 학습 손실 기록만 고릅니다.
eval_log = [(history["epoch"], history["eval_loss"]) for history in trainer.state.log_history if "eval_loss" in history]  # 검증 손실 기록만 고릅니다.
plt.figure(figsize=(7, 3.5))
plt.plot(*zip(*train_log), marker="o", markersize=3, label="학습 손실 (train)")
plt.plot(*zip(*eval_log), marker="s", label="검증 손실 (valid)")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("페르소나 LoRA 학습 곡선")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
pd.DataFrame(eval_log, columns=["epoch", "eval_loss"])

### 5.6 학습 전후 응답 비교

In [ ]:
lora_model = trainer.model  # 학습이 끝난 LoRA 모델입니다.
after_answer_1 = chat(lora_model, question_1)
after_answer_2 = chat(lora_model, question_2)
print(f"Q: {question_1}\n[학습 전] {before_answer_1}\n[학습 후] {after_answer_1}\n")
print(f"Q: {question_2}\n[학습 전] {before_answer_2}\n[학습 후] {after_answer_2}")

학습 데이터에 없던 질문에도 **목표 말투가 일반화**되는지 확인하세요.  
0.5B 모델은 크기가 작아 문장이 어색하거나 내용이 엉뚱할 수 있습니다. 여기서는 **말투 변화**를 확인합니다. 답변 내용의 정확성은 별도로 평가해야 합니다.

### 5.7 어댑터 저장

`save_pretrained()`는 기반 모델 전체가 아니라 **LoRA 어댑터만** 저장합니다.

In [ ]:
# 학습한 LoRA 어댑터를 저장하고 파일이 얼마나 작은지 확인합니다.
ADAPTER_DIR = "outputs/persona-lora-adapter"  # 어댑터와 토크나이저를 저장할 폴더입니다.
lora_model.save_pretrained(ADAPTER_DIR)  # LoRA A·B 행렬과 설정만 저장합니다.
tokenizer.save_pretrained(ADAPTER_DIR)  # 같은 토크나이저 설정도 저장합니다.

for name in sorted(os.listdir(ADAPTER_DIR)):  # 저장된 파일 이름을 차례로 확인합니다.
    size = os.path.getsize(os.path.join(ADAPTER_DIR, name))  # 현재 파일의 바이트 수입니다.
    print(f"{name:<28} {size / 1024**2:8.2f} MB")

base_params = sum(p.numel() for name, p in model.named_parameters() if "lora_" not in name)  # LoRA를 뺀 기반 모델 파라미터 수입니다.
print(f"\n참고: 기반 모델 가중치(FP32) 크기 약 {base_params * 4 / 1024**2:,.0f} MB")
print(json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))["base_model_name_or_path"], "<- 어댑터가 기억하는 기반 모델")

- `adapter_model.safetensors`: 학습한 LoRA 행렬 ($A$, $B$)  
- `adapter_config.json`: `r`, `lora_alpha`, `target_modules`, **기반 모델 이름**  
- 토크나이저 파일: 템플릿과 특수 토큰 정보 (어댑터와 함께 보관하면 편리)

어댑터는 **자신이 학습된 기반 모델과 함께 써야** 합니다. 다른 모델에 붙이면 동작하지 않습니다.

### 5.8 학습이 잘 안 될 때 점검표

| 증상 | 의심할 원인 | 점검 방법 |
|------|-------------|-----------|
| 학습 후에도 응답이 거의 그대로 | 학습률이 너무 낮음, epoch 부족, 추론 템플릿 불일치 | 손실 곡선, `chat()`이 `apply_chat_template`을 쓰는지 |
| 답변을 끝내지 못하고 계속 생성 | 학습 데이터에 종료 토큰 없음, `max_length`에서 잘림 | 토큰화한 학습 예시 끝에 `<|im_end|>`이 남는지 확인 |
| 질문을 따라 하거나 이상한 기호 출력 | labels 마스킹 오류, 데이터에 이상한 기호가 섞여 있었을 가능성 | `completion_mask`, 학습 데이터 원문 확인 |
| 학습 데이터 문장을 그대로 외워서 출력 | 과적합 | 검증 손실 추이, epoch·r 줄이기 |
| 원래 잘하던 일반 질문을 못함 | 과도한 학습으로 기존 능력 손상 | 6.3절처럼 일반 질문으로 확인 |
| 메모리 부족 (CUDA out of memory, RAM 부족) | 배치·max_length가 큼 | 배치 줄이고 `gradient_accumulation_steps` 늘리기, QLoRA, gradient checkpointing |

## 6. 모델 평가와 활용

### 6.1 무엇을 평가할까

파인튜닝한 모델은 **여러 관점**에서 확인해야 합니다. 손실이 줄었다고 해서 원하는 모델이 되었다는 보장은 없습니다.

| 관점 | 방법 | 예 |
|------|------|-----|
| 언어 모델 지표 | 평가 데이터의 손실 | 낮을수록 정답 답변을 더 잘 예측 |
| 목표 달성도 | 규칙 기반 자동 채점 | 형식 준수율, 분류 정확도, 말투 규칙 준수율 |
| 품질 | 사람 평가, LLM-as-a-Judge (더 큰 LLM이 채점) | 유용성, 자연스러움 |
| 부작용 | 학습 목표와 무관한 질문으로 확인 | 일반 능력 저하, 안전성, 환각 |

5.6절에서는 학습 데이터에 없는 질문 두 개로 말투 변화를 확인했습니다. 이 장에서는 **평가 데이터 전체**로 정량 지표까지 확인합니다.

### 6.2 실습: 손실 비교

PEFT 모델은 `disable_adapter()`로 **어댑터를 잠시 끈 상태**, 즉 원래 모델로 계산할 수 있습니다.  
같은 평가 데이터(test, 학습에 쓰지 않은 데이터)에서 학습 전후 손실을 비교합니다.

아래 `build_features()`는 test 손실을 계산할 때 질문 부분을 제외하기 위한 평가용 보조 함수입니다.  
정답이 시작되기 전까지 `labels`를 `-100`으로 가리고, 답변 토큰의 평균 손실을 계산합니다.

In [ ]:
# 평가 데이터의 질문과 답변을 모델 입력과 정답 토큰으로 바꿉니다.
def build_features(prompt, response):
    prompt_messages = [{"role": "user", "content": prompt}]  # 질문만 있는 메시지입니다.
    full_messages = prompt_messages + [{"role": "assistant", "content": response}]  # 정답까지 포함한 메시지입니다.

    prompt_ids = tokenizer.apply_chat_template(prompt_messages, add_generation_prompt=True)  # 정답이 시작되기 전 토큰입니다.
    full_ids = tokenizer.apply_chat_template(full_messages)  # 질문과 정답 전체의 토큰입니다.

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]  # 질문 부분을 -100으로 표시해 손실 계산에서 제외합니다.
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}  # 평가에 필요한 세 배열을 반환합니다.

In [ ]:
# 모델이 정답 답변의 각 토큰을 얼마나 잘 예측했는지 평균 손실로 계산합니다.
from torch.nn.utils.rnn import pad_sequence  # 길이가 다른 예시를 패딩해 배치로 묶습니다.

def evaluate_loss(model, dataset, batch_size=1):  # 같은 평가 절차를 기반 모델과 LoRA 모델에 적용합니다.
    features = [build_features(row["prompt"][0]["content"], row["completion"][0]["content"]) for row in dataset]  # 모든 대화를 토큰 배열로 바꿉니다.
    pad_values = {"input_ids": tokenizer.pad_token_id, "attention_mask": 0, "labels": -100}  # 입력은 패딩 토큰, 마스크는 0, 정답은 -100으로 채웁니다.
    total_loss, total_tokens = 0.0, 0  # 전체 손실과 정답 토큰 수를 누적합니다.
    model.eval()  # 평가 모드로 전환합니다.
    for start in range(0, len(features), batch_size):  # 평가 예시를 배치 크기만큼 나눕니다.
        chunk = features[start:start + batch_size]
        batch = {  # 길이가 다른 배열을 패딩해 배치로 묶습니다.
            key: pad_sequence([torch.tensor(feature[key]) for feature in chunk], batch_first=True, padding_value=pad).to(model.device)  # 현재 항목을 모델 장치의 텐서로 만듭니다.
            for key, pad in pad_values.items()  # input_ids·mask·labels에 같은 작업을 반복합니다.
        }
        with torch.inference_mode():  # 평가에서는 그래디언트를 계산하지 않습니다.
            out = model(**batch)  # 현재 배치의 손실과 로짓을 얻습니다.
        n_tokens = (batch["labels"][:, 1:] != -100).sum().item()  # 다음 토큰 예측에서 첫 위치를 제외하고 정답 토큰 수를 셉니다.
        total_loss += out.loss.item() * n_tokens  # 평균 손실을 토큰 수로 가중해 더합니다.
        total_tokens += n_tokens  # 전체 정답 토큰 수를 더합니다.
    return total_loss / total_tokens  # 답변 토큰 기준 평균 손실을 반환합니다.

In [ ]:
test_dataset = persona_ds["test"]  # 학습에 쓰지 않은 평가 데이터입니다.
lora_loss = evaluate_loss(lora_model, test_dataset)  # LoRA 모델의 손실입니다.
with lora_model.disable_adapter():  # 이 블록 안에서는 LoRA를 끄고 원래 모델로 계산합니다.
    base_loss = evaluate_loss(lora_model, test_dataset)  # 같은 모델에서 어댑터만 끈 손실입니다.

eval_table = pd.DataFrame({  # 전후 손실을 나란히 비교할 표입니다.
    "학습 전 (어댑터 끔)": [base_loss],  # 기반 모델 손실입니다.
    "학습 후 (LoRA)": [lora_loss],  # LoRA 모델 손실입니다.
}, index=["test loss"]).round(3)
eval_table

손실이 낮을수록 **처음 보는 질문에서도 목표 말투의 답변을 더 잘 예측**한다는 뜻입니다. 5.6절에서 직접 읽은 답변과 같은 결론입니다  
— 학습 데이터를 단순히 외운 게 아니라 처음 보는 질문에도 일반화된 것입니다.

다만 이 손실은 **정답 텍스트를 얼마나 잘 예측하는지**를 재는 값입니다. 정답을 입력에 넣어 둔 채 다음 토큰을 맞히게 하는 방식이라, 뜻과 말투가 같아도 **표현이 다르면 손실이 높게 나옵니다.** 답변이 옳은지를 재는 지표가 아닙니다.

그래서 이 표는 **같은 데이터에서 학습 전후를 비교하는 상대 지표**로만 봅니다. "제대로 답했는가"는 5.6절처럼 답변을 직접 읽고, 6.1절 표의 규칙 기반 채점이나 사람 평가로 확인합니다.

### 6.3 실습: 부작용 확인 - 일반 질문

좁은 데이터로 학습하면 **학습 목표와 무관한 질문에도 영향**이 갈 수 있습니다.  
이 실습의 페르소나는 일부러 과장된 말투라 영향이 잘 보입니다. 실제 서비스 모델이라면 이런 변화가 허용 범위인지 반드시 확인해야 합니다.

In [ ]:
# 학습에 사용하지 않은 일반 질문으로 기반 모델과 LoRA 모델의 답변을 비교합니다.
general_questions = ["대한민국의 수도는 어디인가요?", "3 더하기 5는 얼마인가요?", "물은 몇 도에서 끓나요?"]
for question in general_questions:
    with lora_model.disable_adapter():  # LoRA 어댑터를 잠시 끄고 기반 모델의 답변을 생성합니다.
        before = chat(lora_model, question, max_new_tokens=40)  # 기반 모델 답변을 생성합니다.
    after = chat(lora_model, question, max_new_tokens=40)  # LoRA 적용 답변을 생성합니다.
    print(f"Q: {question}\n  학습 전: {before[:80]}\n  학습 후: {after[:80]}\n")

학습 후에는 일반 질문에도 페르소나 말투가 나타나거나, **정답 대신 비꼬는 답변**이 나올 수 있습니다. 학습 데이터에 질문에 직접 답하지 않는 예시가 많기 때문입니다.

**실무에서의 대응**  
- 목표 행동 데이터에 **일반 질문에 정확히 답하는 예시**를 함께 섞습니다.  
- 학습 epoch과 학습률을 줄여 기존 능력을 덜 건드립니다.  
- 필요할 때만 어댑터를 켜는 방식(다음 절)으로 운영합니다.

### 6.4 실습: 저장한 어댑터 불러오기, 끄고 켜기, 병합하기

실제 서비스에서는 저장해 둔 어댑터를 불러와 씁니다. 어댑터를 붙인 채 쓰면(`PeftModel.from_pretrained`) 업무별로 자유롭게 교체·끄기가 가능하고, `merge_and_unload()`로 병합하면 일반 모델과 같은 속도로 배포할 수 있는 대신 더는 어댑터를 떼어낼 수 없습니다.

In [ ]:
# 저장한 LoRA 어댑터를 새로 불러온 기반 모델에 연결합니다.
from peft import PeftModel  # 저장한 어댑터를 기반 모델에 붙입니다.

del trainer, lora_model, model  # 학습에 쓴 모델 객체를 지웁니다.
gc.collect()  # 파이썬 객체를 정리합니다.
if torch.cuda.is_available():  # CUDA GPU를 사용 중이면 캐시를 비웁니다.
    torch.cuda.empty_cache()  # GPU 캐시도 비웁니다.

base = load_base_model()  # 같은 기반 모델을 새로 불러옵니다.
loaded = PeftModel.from_pretrained(base, ADAPTER_DIR)  # 저장한 LoRA 어댑터를 붙입니다.

question = "이번 주말에 등산을 가려는데 날씨가 걱정돼."  # 어댑터 효과를 볼 새 질문입니다.
print("[어댑터 켬]", chat(loaded, question))  # LoRA 적용 답변을 생성합니다.
with loaded.disable_adapter():  # 이 블록에서 어댑터를 잠시 끕니다.
    print("[어댑터 끔]", chat(loaded, question)[:150])

In [ ]:
# LoRA가 학습한 변화량을 기반 모델에 합친 뒤 전체 모델을 저장합니다.
merged_model = loaded.merge_and_unload()  # LoRA 변화량을 기반 가중치에 합칩니다.
print("병합 후 모델 종류:", type(merged_model).__name__)
print("[병합 모델]", chat(merged_model, question))

MERGED_DIR = "outputs/persona-merged"  # 병합 모델을 저장할 폴더입니다.
merged_model.save_pretrained(MERGED_DIR)  # 전체 가중치 모델로 저장합니다.
merged_size = sum(os.path.getsize(os.path.join(MERGED_DIR, name)) for name in os.listdir(MERGED_DIR))  # 병합 모델 파일 크기를 합칩니다.
adapter_size = sum(os.path.getsize(os.path.join(ADAPTER_DIR, name)) for name in os.listdir(ADAPTER_DIR) if name.startswith("adapter"))  # 어댑터 파일 크기를 합칩니다.
print(f"\n어댑터 {adapter_size / 1024**2:.1f} MB vs 병합 모델 {merged_size / 1024**2:,.0f} MB")

### 6.5 서비스에 활용하는 방법

| 방식 | 설명 |
|------|------|
| 다중 어댑터 운영 | 하나의 기반 모델에 업무별 LoRA(상담, 요약, 분류 등)를 두고 요청마다 바꿔 끼움. vLLM, TGI 같은 서빙 엔진이 여러 LoRA 동시 서빙을 지원 |
| 병합 모델 배포 | `merge_and_unload()` 후 일반 모델처럼 배포. 필요하면 GGUF로 변환해 llama.cpp·Ollama로 로컬 실행 |
| Hub 공유 | `push_to_hub()`로 어댑터만 올려 팀과 공유 (`Write` 토큰 필요, 기반 모델 라이선스 확인) |
| RAG와 조합 | 파인튜닝으로 형식·말투·절차를 익히고, 사실 정보는 RAG로 제공 |
| 지속 개선 | 실제 사용 로그에서 실패 사례를 모아 SFT 데이터에 추가하고 재학습 |

## 7. (선택) QLoRA로 기반 모델까지 줄이기

LoRA로 학습하고, 평가하고, 어댑터를 저장·병합하는 과정까지 모두 마쳤습니다.  
여기서는 기반 모델 자체를 4비트로 줄이는 **QLoRA**를 살펴보고, 같은 데이터로 학습을 한 번 더 해 봅니다.  
시간이 부족하면 이 장은 건너뛰고 8장으로 넘어가도 됩니다.

### 7.1 QLoRA의 원리

LoRA는 학습할 파라미터를 줄이지만, 기반 모델의 가중치는 여전히 **BF16이나 FP16 같은 16비트 정밀도**로 GPU 메모리를 차지합니다.  
[QLoRA(2023)](https://arxiv.org/abs/2305.14314)는 기반 모델을 **4비트로 양자화**해서 이 부담까지 줄입니다.

<img src="image/qlora_concept.svg" width="720">

**양자화(Quantization)** 는 실수 값을 더 적은 비트로 근사해 저장하는 기술입니다. 1.2절 표에서 본 것처럼 FP32는 32비트, BF16은 16비트이고, QLoRA가 쓰는 **NF4는 4비트**(1개당 0.5바이트)입니다.

**QLoRA의 세 가지 핵심**  
1. **NF4 양자화**: 신경망 가중치가 대체로 0 근처에 몰린 정규분포라는 점을 이용해, 16개의 값을 분포에 맞게 배치합니다.  
2. **Double Quantization**: 양자화에 필요한 스케일 값들까지 한 번 더 양자화해 메모리를 더 줄입니다.  
3. **Paged Optimizer**: GPU 메모리가 순간적으로 부족할 때 옵티마이저 상태를 CPU 메모리로 옮겨 오류를 막습니다.

계산할 때는 4비트 가중치를 BF16/FP32로 **잠깐 되돌려(역양자화)** 쓰고, LoRA 행렬은 처음부터 높은 정밀도로 학습합니다.  
그래서 메모리는 크게 줄지만, 양자화·역양자화 때문에 학습 속도는 LoRA보다 **조금 느려질 수 있습니다.**

**바로 확인하기**

QLoRA에서 학습되는 파라미터는 4비트일까요?

<details>
<summary>정답 보기</summary>

아닙니다. 4비트로 저장되는 것은 동결된 기반 모델이고, LoRA 행렬 $A$, $B$는 BF16이나 FP32 같은 높은 정밀도로 학습합니다.

</details>

### 7.2 실습: 정밀도별 메모리 비교 (CUDA GPU 전용)

같은 Qwen2.5-0.5B-Instruct을 FP32, BF16, 4비트(NF4)로 불러와 메모리를 비교합니다.  
`bitsandbytes`의 4비트 양자화는 CUDA GPU에서만 동작하므로, CPU에서는 아래 셀이 자동으로 건너뛰어집니다.

In [ ]:
# 같은 모델을 서로 다른 숫자 정밀도로 불러와 가중치 메모리 사용량을 비교합니다.
from transformers import BitsAndBytesConfig  # 4비트 양자화 설정 도구를 불러옵니다.

bnb_config = BitsAndBytesConfig(  # 4비트 NF4 양자화 방식을 설정합니다.
    load_in_4bit=True,  # 선형층 가중치를 4비트로 읽습니다.
    bnb_4bit_quant_type="nf4",  # 가중치의 양자화 형식입니다.
    bnb_4bit_use_double_quant=True,  # 양자화 상수도 압축합니다.
    bnb_4bit_compute_dtype=MODEL_DTYPE,  # GPU에 맞는 계산 정밀도입니다.
)

if torch.cuda.is_available():  # 4비트 양자화는 CUDA GPU에서만 실행합니다.
    footprints = {}  # 정밀도별 메모리 사용량을 저장합니다.
    for label, dtype in [("FP32", torch.float32), ("BF16 / FP16", MODEL_DTYPE)]:  # 두 정밀도를 비교합니다.
        precision_model = load_base_model(dtype=dtype)  # 현재 정밀도로 모델을 불러옵니다.
        footprints[label] = precision_model.get_memory_footprint()  # 가중치 메모리를 기록합니다.
        del precision_model  # 비교가 끝난 모델 참조를 삭제합니다.
        gc.collect()  # 파이썬 객체를 정리합니다.
        torch.cuda.empty_cache()  # GPU 캐시를 비웁니다.

    q4_model = load_base_model(dtype=MODEL_DTYPE, quantization_config=bnb_config, device_map={"": 0})  # 같은 모델을 NF4 4비트로 불러옵니다.
    footprints["4bit (NF4)"] = q4_model.get_memory_footprint()  # 4비트 메모리를 기록합니다.
    print(q4_model.model.layers[0].self_attn.q_proj)  # 4비트 선형층의 형태를 확인합니다.
    del q4_model  # 4비트 비교 모델 참조를 삭제합니다.
    gc.collect()  # 파이썬 객체를 정리합니다.
    torch.cuda.empty_cache()  # 다음 실습을 위해 GPU 캐시를 비웁니다.
    display(pd.Series({label: f"{footprint / 1024**2:,.0f} MB" for label, footprint in footprints.items()}, name="메모리"))
else:  # CUDA GPU가 없으므로 이 실습을 건너뜁니다.
    print("CUDA GPU가 없어 4비트 양자화 실습을 건너뜁니다.")

4비트 모델의 전체 메모리가 FP32의 1/8이 되는 것은 아닙니다. `bitsandbytes`는 **선형층을 양자화**하고 임베딩 같은 층은 높은 정밀도로 유지합니다. Qwen2.5-0.5B-Instruct는 임베딩이 전체 파라미터의 약 28%이고, 양자화 상수도 추가되므로 실제 측정값을 확인해야 합니다.  
7B 이상 모델에서는 대부분의 파라미터가 선형층에 있어 QLoRA의 절감 효과가 훨씬 큽니다.

| 구분 | Full Fine-tuning | LoRA | QLoRA |
|------|------------------|------|-------|
| 기반 모델 | 학습 (FP32/BF16) | 동결 (BF16) | 동결 (4bit) |
| 학습 파라미터 | 전체 | 1% 내외 | 1% 내외 |
| GPU 메모리 | 매우 큼 | 중간 | 작음 |
| 학습 속도 | 느림 | 빠름 | LoRA보다 약간 느릴 수 있음 |
| 결과물 | 전체 모델 (GB 단위) | 어댑터 (MB 단위) | 어댑터 (MB 단위) |
| 적합한 상황 | 자원이 충분하고 큰 변화가 필요 | 대부분의 업무 적응 | GPU 메모리가 부족할 때, 큰 모델 |

### 7.3 QLoRA로 같은 학습 해보기 (CUDA GPU 전용)

기반 모델을 4비트로 불러와 **같은 데이터, 같은 LoRA 설정**으로 학습합니다. CUDA GPU가 없으면 아래 셀들이 자동으로 건너뛰어집니다.  
`prepare_model_for_kbit_training()`은 양자화 모델을 학습하기 좋게 준비합니다(정규화층을 FP32로 유지하고, 입력 임베딩에서 그래디언트가 흐르도록 설정).

In [ ]:
# LoRA 학습 조건을 복사하고 QLoRA 결과의 저장 경로만 바꿉니다.
import copy  # 학습 설정을 안전하게 복사합니다.

q_sft_config = copy.deepcopy(sft_config)  # LoRA와 같은 학습 조건입니다.
q_sft_config.output_dir = "outputs/persona-qlora"  # QLoRA 결과는 다른 폴더에 저장합니다.


In [ ]:
# 4비트로 압축한 기반 모델에 LoRA를 붙여 학습합니다.
from peft import prepare_model_for_kbit_training  # 양자화 모델을 학습용으로 준비합니다.

if torch.cuda.is_available():  # 4비트 양자화는 CUDA GPU에서만 동작합니다.
    q_model = load_base_model(dtype=MODEL_DTYPE, quantization_config=bnb_config, device_map={"": 0})  # 4비트 기반 모델을 불러옵니다.
    q_model = prepare_model_for_kbit_training(q_model)  # 양자화 모델을 학습 가능하게 준비합니다.

    q_trainer = SFTTrainer(  # 4비트 기반 모델용 Trainer를 만듭니다.
        model=q_model,  # 양자화한 기반 모델입니다.
        args=q_sft_config,  # 바로 위 셀의 QLoRA 설정입니다.
        train_dataset=persona_ds["train"],  # 학습 데이터입니다.
        eval_dataset=persona_ds["valid"],  # 검증 데이터입니다.
        processing_class=tokenizer,  # 템플릿 적용과 토큰화를 맡습니다.
        peft_config=lora_config,  # 같은 LoRA 설정을 붙입니다.
    )
else:  # CUDA GPU가 없으므로 이 실습을 건너뜁니다.
    print("CUDA GPU가 없어 QLoRA 실습을 건너뜁니다.")

In [ ]:
if torch.cuda.is_available():  # 위 셀에서 QLoRA Trainer를 만들었을 때만 학습합니다.
    q_trainer.train()  # 4비트 기반 모델에 LoRA를 학습합니다.

    qlora_eval_losses = [history["eval_loss"] for history in q_trainer.state.log_history if "eval_loss" in history]  # QLoRA 검증 손실 목록입니다.
    summary = pd.DataFrame({  # 두 방식의 마지막 검증 손실을 비교합니다.
        "LoRA": [round(eval_log[-1][1], 3)],  # 5.5절에서 기록한 LoRA의 마지막 검증 손실입니다.
        "QLoRA": [round(qlora_eval_losses[-1], 3)],  # 4비트 QLoRA 결과입니다.
    }, index=["마지막 검증 손실"])
    display(summary)
    print("QLoRA 학습 후:", chat(q_trainer.model, "업무 실수를 줄이려면 체크리스트를 쓰는 게 좋을까?"))

In [ ]:
if torch.cuda.is_available():  # QLoRA를 실행했다면 정리합니다.
    del q_trainer, q_model  # 양자화 모델과 Trainer 참조를 지웁니다.
    gc.collect()  # 파이썬 객체를 정리합니다.
    torch.cuda.empty_cache()  # GPU 캐시를 비웁니다.

작은 모델에서는 QLoRA의 메모리 절감 효과가 작고, 양자화·역양자화 때문에 학습이 더 느릴 수도 있습니다.  
QLoRA는 **7B·13B·70B처럼 16비트 가중치를 GPU에 올리기 어려운 큰 모델**을 학습할 때 특히 유용합니다.

## 8. 종합평가 안내

종합평가는 별도 노트북에서 진행합니다.

- 파일: `최종과제.ipynb`  
- 주제: **고객 문의 응답 형식 LoRA 파인튜닝**  
- 내용: 고객 문의를 `분류 / 긴급도 / 답변` 세 줄 형식으로 답하도록 Qwen2.5-0.5B-Instruct을 LoRA로 학습하고, 학습 전후를 정량 평가합니다.  
- 핵심 요소: SFT 데이터 변환, Chat Template, LoraConfig·SFTConfig 설정, 학습 전후 평가  
- 제출: 노트북 안내에 따라 실행 결과 4가지를 캡처해 `수행평가 제출양식_본인이름.docx`에 붙여 제출합니다.

2일차 본편을 마친 뒤 `최종과제.ipynb`를 열어 안내 순서대로 실행하세요.

## 9. 2일차 정리

| 주제 | 핵심 포인트 |
|------|-------------|
| 모델 준비 | Qwen 불러오기, 정밀도별 가중치 메모리 = 파라미터 수 × 바이트 수 |
| SFT 데이터 | (요청, 모범 답변) 쌍. 대화 형식 변환 -> train/valid/test 분리 |
| Chat Template | messages를 모델별 문자열 형식으로 변환. 학습과 추론에서 같은 템플릿 사용 |
| labels | 질문 구간은 -100으로 가리고 답변과 종료 토큰만 학습 |
| PEFT | 기반 모델은 동결하고 일부 파라미터만 학습해 메모리·저장 공간 절감 |
| LoRA | 변화량을 저랭크 행렬 $A$, $B$로 학습. $B=0$ 초기화, 병합 가능 |
| QLoRA | 4비트 기반 모델 + LoRA. 큰 모델을 GPU 한 장으로 학습 |
| SFTTrainer | 템플릿 적용, 토큰화, 답변 마스킹, 학습 루프를 한 번에 처리 |
| 평가 | 손실(정량) + 답변을 직접 읽어보는 정성 비교 + 부작용 점검을 함께 확인 |
| 활용 | 어댑터 교체 운영 또는 병합 배포, RAG와 조합 |

```text
데이터 품질 -> 결과를 결정
Chat Template + labels 마스킹 -> 모델이 "무엇을" 배우는지 결정
LoRA 설정 (r, target_modules, 학습률, epoch) -> "얼마나" 배우는지 결정
평가 (정량 + 부작용) -> 서비스에 써도 되는지 판단
```